In [ ]:

import json
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

# =========================
# Config
# =========================
MATSCHOLAR_JSON = "matscholar-embedding.json"
INPUT_FILE      = "SMILES_original.csv"
OUTPUT_FILE     = "polymer_matscholar_env_vectors.csv"
FAIL_FILE       = "matscholar_env_failures.csv"

INCLUDE_HYDROGENS = False   # Usually False for environments (Hs can dominate counts)
RADIUS = 2                  # 1 or 2 are typical


# ============================================================
# Load MatScholar embedding table (element -> vector)
# ============================================================
with open(MATSCHOLAR_JSON, "r", encoding="utf-8") as f:
    elem_to_vec_raw = json.load(f)

elem_to_vec = {}
dims = set()

for k, v in elem_to_vec_raw.items():
    key = str(k).strip()
    arr = np.asarray(v, dtype=np.float32)
    if arr.ndim != 1:
        raise ValueError(f"Embedding for element '{key}' is not 1D (shape={arr.shape}).")
    elem_to_vec[key] = arr
    dims.add(arr.shape[0])

if len(dims) != 1:
    raise ValueError(f"Inconsistent embedding dimensions found: {sorted(dims)}")

DIM = dims.pop()
print(f"Loaded MatScholar embeddings for {len(elem_to_vec)} elements. dim={DIM}")

if INCLUDE_HYDROGENS and "H" not in elem_to_vec:
    print(
        "Warning: INCLUDE_HYDROGENS=True but 'H' not present in embeddings. "
        "Hydrogens will be counted as unknown and ignored."
    )


# ============================================================
# Load SMILES CSV
# ============================================================
def load_smiles_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.lower().strip() for c in df.columns]

    if "smiles" not in df.columns:
        cols = list(df.columns)
        if len(cols) < 2:
            raise ValueError("INPUT_FILE must contain at least 2 columns (name, smiles) or a 'smiles' column.")
        df.columns = ["name", "smiles"] + cols[2:]

    if "name" not in df.columns:
        df["name"] = ""

    df = df.dropna(subset=["smiles"]).copy()
    df["smiles"] = df["smiles"].astype(str).str.strip()
    df["name"] = df["name"].astype(str).str.strip()
    return df


# ============================================================
# Graph neighbourhood utilities
# ============================================================
def get_shell_indices(mol: Chem.Mol, center_idx: int, radius: int):
    """
    shells[k] = set(atom indices at graph distance exactly k from center_idx)
    k = 1..radius
    """
    visited = {center_idx}
    frontier = {center_idx}
    shells = {}

    for k in range(1, radius + 1):
        next_frontier = set()
        for idx in frontier:
            atom = mol.GetAtomWithIdx(idx)
            for nbr in atom.GetNeighbors():
                j = nbr.GetIdx()
                if j not in visited:
                    next_frontier.add(j)
        shells[k] = next_frontier
        visited |= next_frontier
        frontier = next_frontier

    return shells


def agg_stats(vecs: list[np.ndarray], dim: int):
    """Return (mean, std) for list of vectors; zeros if empty."""
    if not vecs:
        z = np.zeros((dim,), dtype=np.float32)
        return z, z
    V = np.vstack(vecs).astype(np.float32)
    return V.mean(axis=0), V.std(axis=0)


# ============================================================
# Local scalar atom features (compact, environment-relevant)
# ============================================================
def atom_local_scalar_features(atom: Chem.Atom) -> np.ndarray:
    hyb = atom.GetHybridization()
    hyb_sp    = 1.0 if hyb == Chem.rdchem.HybridizationType.SP else 0.0
    hyb_sp2   = 1.0 if hyb == Chem.rdchem.HybridizationType.SP2 else 0.0
    hyb_sp3   = 1.0 if hyb == Chem.rdchem.HybridizationType.SP3 else 0.0
    hyb_other = 1.0 if (hyb_sp + hyb_sp2 + hyb_sp3) == 0.0 else 0.0

    return np.array([
        float(atom.GetDegree()),
        float(atom.GetTotalDegree()),
        float(atom.GetExplicitValence()),
        float(atom.GetImplicitValence()),
        float(atom.GetFormalCharge()),
        float(atom.GetTotalNumHs()),
        1.0 if atom.GetIsAromatic() else 0.0,
        1.0 if atom.IsInRing() else 0.0,
        hyb_sp, hyb_sp2, hyb_sp3, hyb_other
    ], dtype=np.float32)


# ============================================================
# Per-atom environment vector builder
# ============================================================
def atom_environment_vector(
    mol: Chem.Mol,
    atom_idx: int,
    elem_to_vec: dict,
    dim: int,
    radius: int = 2
):
    """
    Environment vector for one atom/site:
      [center_embed,
       shell1_mean, shell1_std,
       shell2_mean, shell2_std (if radius>=2),
       local_scalar_features]
    """
    atom = mol.GetAtomWithIdx(atom_idx)
    sym = atom.GetSymbol().strip()

    center = elem_to_vec.get(sym)
    if center is None:
        return None, {"unknown_center": True}

    shells = get_shell_indices(mol, atom_idx, radius=radius)

    # 1-hop shell
    shell1_vecs = []
    unknown1 = 0
    for j in shells.get(1, set()):
        v = elem_to_vec.get(mol.GetAtomWithIdx(j).GetSymbol().strip())
        if v is None:
            unknown1 += 1
        else:
            shell1_vecs.append(v)
    s1_mean, s1_std = agg_stats(shell1_vecs, dim)

    parts = [center, s1_mean, s1_std]

    # 2-hop shell
    unknown2 = 0
    if radius >= 2:
        shell2_vecs = []
        for j in shells.get(2, set()):
            v = elem_to_vec.get(mol.GetAtomWithIdx(j).GetSymbol().strip())
            if v is None:
                unknown2 += 1
            else:
                shell2_vecs.append(v)
        s2_mean, s2_std = agg_stats(shell2_vecs, dim)
        parts += [s2_mean, s2_std]

    # Local RDKit scalars
    parts.append(atom_local_scalar_features(atom))

    x = np.concatenate(parts).astype(np.float32)
    stats = {
        "unknown_center": False,
        "unknown_shell1": unknown1,
        "unknown_shell2": unknown2,
        "deg": int(atom.GetDegree()),
        "radius": radius,
    }
    return x, stats


# ============================================================
# Main run: produce one row per atom environment
# ============================================================
def main():
    df = load_smiles_csv(INPUT_FILE)

    rows = []
    failed = []

    total_mols = 0
    parse_fail = 0
    total_atoms = 0
    unknown_center_atoms = 0
    unknown_shell1_total = 0
    unknown_shell2_total = 0

    for name, smi in df[["name", "smiles"]].itertuples(index=False):
        total_mols += 1
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            parse_fail += 1
            failed.append((name, smi, "parse_fail"))
            continue

        if INCLUDE_HYDROGENS:
            try:
                mol = Chem.AddHs(mol)
            except Exception:
                parse_fail += 1
                failed.append((name, smi, "addHs_fail"))
                continue

        for atom in mol.GetAtoms():
            total_atoms += 1
            atom_idx = atom.GetIdx()
            sym = atom.GetSymbol()

            x, stats = atom_environment_vector(
                mol, atom_idx, elem_to_vec, DIM, radius=RADIUS
            )

            if x is None:
                unknown_center_atoms += 1
                continue

            unknown_shell1_total += stats["unknown_shell1"]
            unknown_shell2_total += stats["unknown_shell2"]

            rows.append({
                "name": name,
                "smiles": smi,
                "atom_index": atom_idx,
                "atom_symbol": sym,
                "degree": stats["deg"],
                "radius": stats["radius"],
                "unknown_shell1": stats["unknown_shell1"],
                "unknown_shell2": stats["unknown_shell2"],
                "vec": x
            })

    print(f"Total molecules: {total_mols} | parse/addHs failures: {parse_fail}")
    print(f"Total atoms visited: {total_atoms}")
    print(f"Unknown-center atoms skipped: {unknown_center_atoms}")
    print(f"Unknown neighbour counts: shell1={unknown_shell1_total}, shell2={unknown_shell2_total}")

    if not rows:
        raise RuntimeError("No atom environments were featurised. Check SMILES and embedding coverage.")

    X = np.vstack([r["vec"] for r in rows])
    feat_cols = [f"matscholar_env_{i}" for i in range(X.shape[1])]

    out = pd.DataFrame({
        "name": [r["name"] for r in rows],
        "smiles": [r["smiles"] for r in rows],
        "atom_index": [r["atom_index"] for r in rows],
        "atom_symbol": [r["atom_symbol"] for r in rows],
        "degree": [r["degree"] for r in rows],
        "radius": [r["radius"] for r in rows],
        "unknown_shell1": [r["unknown_shell1"] for r in rows],
        "unknown_shell2": [r["unknown_shell2"] for r in rows],
    })
    out = pd.concat([out, pd.DataFrame(X, columns=feat_cols)], axis=1)
    out.to_csv(OUTPUT_FILE, index=False)

    print("Saved:", OUTPUT_FILE, "shape:", X.shape)

    if failed:
        pd.DataFrame(failed, columns=["name", "smiles", "reason"]).to_csv(FAIL_FILE, index=False)
        print("Saved failures:", FAIL_FILE, "n=", len(failed))


if __name__ == "__main__":
    main()



Loaded MatScholar embeddings for 103 elements. dim=200
Total molecules: 111 | parse/addHs failures: 7
Total atoms visited: 1052
Unknown-center atoms skipped: 0
Unknown neighbour counts: shell1=0, shell2=0
Saved: polymer_matscholar_env_vectors.csv shape: (1052, 1012)
Saved failures: matscholar_env_failures.csv n= 7
